<a href="https://colab.research.google.com/github/MateusA-Borges/image-classification-project-senac-UC13/blob/main/Intel_Classification_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BASELINE MODEL (1 EPOCH)

In [ ]:
# =========================================================
# 1. Imports
# =========================================================
import os
import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

# =========================================================
# 2. Config
# =========================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMG_SIZE = 150
BATCH_SIZE = 128
EPOCHS = 1          # FAST submission
LR = 3e-4
NUM_CLASSES = 6

# =========================================================
# 3. Locate Dataset
# =========================================================
BASE_PATH = '/content/senac-uc-13-2025-intel-classification-dataset' # Change to wherever your dataset is


print("Using dataset at:", BASE_PATH)

TRAIN_CSV = os.path.join(BASE_PATH, "train ids.csv")
TEST_CSV  = os.path.join(BASE_PATH, "test ids.csv")

TRAIN_DIR = os.path.join(BASE_PATH, "train images")
TEST_DIR  = os.path.join(BASE_PATH, "test images")

# =========================================================
# 4. Load CSVs
# =========================================================
df = pd.read_csv(TRAIN_CSV)
df_test = pd.read_csv(TEST_CSV)

# =========================================================
# 5. Stratified Split
# =========================================================
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["class"],
    random_state=42
)

# =========================================================
# 6. Augmentations
# =========================================================
train_tfms = A.Compose([
    A.RandomResizedCrop(
        size=(IMG_SIZE, IMG_SIZE),
        scale=(0.8, 1.0),
        ratio=(0.75, 1.33)
    ),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(0.2, 0.2, 0.2, 0.2),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2()
])

val_tfms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2()
])

# =========================================================
# 7. Dataset
# =========================================================
class SceneDataset(Dataset):
    def __init__(self, df, img_dir, transforms=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        file_id = self.df.loc[idx, "file_id"]
        img_path = os.path.join(self.img_dir, f"{file_id}.jpg")

        image = cv2.imread(img_path)

        # Skip missing images safely
        if image is None:
            return self.__getitem__((idx + 1) % len(self.df))

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms:
            image = self.transforms(image=image)["image"]

        if "class" in self.df.columns:
            return image, self.df.loc[idx, "class"]

        return image, file_id

# =========================================================
# 8. Dataset Instances + DataLoaders
# =========================================================
train_ds = SceneDataset(train_df, TRAIN_DIR, train_tfms)
val_ds   = SceneDataset(val_df, TRAIN_DIR, val_tfms)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# =========================================================
# 9. Model (offline-safe)
# =========================================================
model = timm.create_model(
    "efficientnet_b3",
    pretrained=False,   # no internet needed
    num_classes=NUM_CLASSES
).to(DEVICE)

# =========================================================
# 10. Loss + Optimizer
# =========================================================
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(df["class"]),
    y=df["class"]
)
weights = torch.tensor(weights, dtype=torch.float).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

# =========================================================
# 11. Training + Validation (FAST)
# =========================================================
best_f1 = 0.0

for epoch in range(EPOCHS):
    model.train()
    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()

    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            preds.extend(outputs.argmax(1).cpu().numpy())
            targets.extend(labels.numpy())

    f1 = f1_score(targets, preds, average="macro")
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Macro F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "best_model.pth")

print("Best Validation F1:", best_f1)

# =========================================================
# 12. Inference on Test Set (ORDER-SAFE)
# =========================================================
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

predictions = []

with torch.no_grad():
    for _, row in df_test.iterrows():
        file_id = row["file_id"]
        img_path = os.path.join(TEST_DIR, f"{file_id}.jpg")

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = val_tfms(image=image)["image"]
        image = image.unsqueeze(0).to(DEVICE)

        output = model(image)
        pred = output.argmax(1).item()

        predictions.append(pred)

# =========================================================
# 13. Create Submission (MATCHES test.csv EXACTLY)
# =========================================================
submission = pd.DataFrame({
    "file_id": df_test["file_id"],
    "class": predictions
})

submission.to_csv("submission.csv", index=False)

print("submission.csv generated correctly")
submission.head()

# BEST MODEL


In [ ]:
# =========================================================
# 1. Imports
# =========================================================
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

# =========================================================
# 2. Config
# =========================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 20
LR = 3e-4
NUM_CLASSES = 6
PATIENCE = 3

# =========================================================
# 3. Locate Dataset
# =========================================================
BASE_PATH = '/content/senac-uc-13-2025-intel-classification-dataset' # Change to wherever your dataset is


TRAIN_CSV = os.path.join(BASE_PATH, "train.csv")
TEST_CSV  = os.path.join(BASE_PATH, "test.csv")
TRAIN_DIR = os.path.join(BASE_PATH, "train")
TEST_DIR  = os.path.join(BASE_PATH, "test")

# =========================================================
# 4. Load Data
# =========================================================
df = pd.read_csv(TRAIN_CSV)
df_test = pd.read_csv(TEST_CSV)

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["class"],
    random_state=42
)

# =========================================================
# 5. Augmentations
# =========================================================
train_tfms = A.Compose([
    A.RandomResizedCrop(
    size=(IMG_SIZE, IMG_SIZE),
    scale=(0.7, 1.0)
)
,
    A.HorizontalFlip(p=0.5),
    A.OneOf([
        A.ColorJitter(0.4, 0.4, 0.4, 0.2),
        A.RandomBrightnessContrast(0.2, 0.2)
    ], p=0.8),
    A.CoarseDropout(
        max_holes=8,
        max_height=IMG_SIZE // 10,
        max_width=IMG_SIZE // 10,
        p=0.5
    ),
    A.Normalize((0.485, 0.456, 0.406),
                (0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_tfms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize((0.485, 0.456, 0.406),
                (0.229, 0.224, 0.225)),
    ToTensorV2()
])

# =========================================================
# 6. Dataset
# =========================================================
class SceneDataset(Dataset):
    def __init__(self, df, img_dir, transforms):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        file_id = self.df.loc[idx, "file_id"]
        img_path = os.path.join(self.img_dir, f"{file_id}.jpg")

        image = cv2.imread(img_path)
        if image is None:
            return self.__getitem__((idx + 1) % len(self.df))

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = self.transforms(image=image)["image"]

        if "class" in self.df.columns:
            return image, self.df.loc[idx, "class"]
        return image, file_id

# =========================================================
# 7. Dataloaders
# =========================================================
train_loader = DataLoader(
    SceneDataset(train_df, TRAIN_DIR, train_tfms),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    SceneDataset(val_df, TRAIN_DIR, val_tfms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# =========================================================
# 8. Model
# =========================================================
model = timm.create_model(
    "efficientnet_b3",
    pretrained=True,
    num_classes=NUM_CLASSES
).to(DEVICE)

# =========================================================
# 9. Loss, Optimizer, Scheduler
# =========================================================
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(df["class"]),
    y=df["class"]
)
weights = torch.tensor(weights, dtype=torch.float).to(DEVICE)

criterion = nn.CrossEntropyLoss(
    weight=weights,
    label_smoothing=0.1
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

# =========================================================
# 10. Training Loop + Early Stopping
# =========================================================
best_f1 = 0
patience_counter = 0
train_losses = []
val_f1s = []


train_losses = []
val_f1s = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    # Loss médio da época
    epoch_loss /= len(train_loader)
    train_losses.append(epoch_loss)

    # ==========================
    # Validação
    # ==========================
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            preds.extend(outputs.argmax(1).cpu().numpy())
            targets.extend(labels.numpy())

    f1 = f1_score(targets, preds, average="macro")
    val_f1s.append(f1)

    scheduler.step()

    print(f"Epoch {epoch+1} | Loss: {epoch_loss:.4f} | Macro F1: {f1:.4f}")

    # ==========================
    # Early stopping
    # ==========================
    if f1 > best_f1:
        best_f1 = f1
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pth")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping")
            break
print("Best Validation F1:", best_f1)

# =========================================================
# 11. Inference + TTA
# =========================================================
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

def predict_tta(model, image):
    preds = []
    for flip in [False, True]:
        img = image.clone()
        if flip:
            img = torch.flip(img, dims=[3])
        preds.append(torch.softmax(model(img), dim=1))
    return torch.mean(torch.stack(preds), dim=0)

predictions = []

with torch.no_grad():
    for _, row in tqdm(df_test.iterrows(), total=len(df_test)):
        file_id = row["file_id"]
        img_path = os.path.join(TEST_DIR, f"{file_id}.jpg")

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = val_tfms(image=image)["image"].unsqueeze(0).to(DEVICE)

        probs = predict_tta(model, image)
        predictions.append(probs.argmax(1).item())

# =========================================================
# 12. Submission
# =========================================================
submission = pd.DataFrame({
    "file_id": df_test["file_id"],
    "class": predictions
})

submission.to_csv("submission.csv", index=False)
print("submission.csv ready")


In [ ]:
## Gráfico — Loss de treino × F1 de validação

import matplotlib.pyplot as plt

epochs = range(1, len(train_losses) + 1)

plt.figure()
plt.plot(epochs, train_losses, label="Training Loss")
plt.plot(epochs, val_f1s, label="Validation Macro F1")
plt.xlabel("Epochs")
plt.ylabel("Value")
plt.title("Training Loss vs Validation Macro F1")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
## Matriz de Confusão (Validação)

from sklearn.metrics import confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

cm = confusion_matrix(targets, preds)
num_classes = cm.shape[0]

plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix - Validation")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.colorbar()

for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, cm[i, j],
                 ha="center", va="center")

plt.tight_layout()
plt.show()


In [ ]:
## F1-score por classe

from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import numpy as np

f1_per_class = f1_score(
    targets,
    preds,
    average=None
)

classes = np.arange(len(f1_per_class))

plt.figure()
plt.bar(classes, f1_per_class)
plt.xlabel("Class")
plt.ylabel("F1-score")
plt.title("F1-score per Class (Validation)")
plt.ylim(0, 1)
plt.grid(axis="y")
plt.show()


In [ ]:
## Distribuição da confiança das predições (probabilidade máxima)
import torch
import matplotlib.pyplot as plt

model.eval()
max_probs = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        outputs = torch.softmax(model(images), dim=1)
        max_prob_batch = outputs.max(dim=1)[0]
        max_probs.extend(max_prob_batch.cpu().numpy())

plt.figure()
plt.hist(max_probs, bins=20)
plt.xlabel("Maximum Predicted Probability")
plt.ylabel("Frequency")
plt.title("Prediction Confidence Distribution (Validation)")
plt.grid(True)
plt.show()


# RESNET MODEL (SLIGHTLY WORSE)

In [ ]:
# =========================================================
# 1. Imports
# =========================================================
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

# =========================================================
# 2. Config
# =========================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 20
LR = 3e-4
NUM_CLASSES = 6
PATIENCE = 3

# =========================================================
# 3. Locate Dataset
# =========================================================
BASE_PATH = '/content/senac-uc-13-2025-intel-classification-dataset' # Change to wherever your dataset is

TRAIN_CSV = os.path.join(BASE_PATH, "train.csv")
TEST_CSV  = os.path.join(BASE_PATH, "test.csv")
TRAIN_DIR = os.path.join(BASE_PATH, "train")
TEST_DIR  = os.path.join(BASE_PATH, "test")

# =========================================================
# 4. Load Data
# =========================================================
df = pd.read_csv(TRAIN_CSV)
df_test = pd.read_csv(TEST_CSV)

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["class"],
    random_state=42
)

# =========================================================
# 5. Augmentations
# =========================================================
train_tfms = A.Compose([
    A.RandomResizedCrop(
    size=(IMG_SIZE, IMG_SIZE),
    scale=(0.7, 1.0)
)
,
    A.HorizontalFlip(p=0.5),
    A.OneOf([
        A.ColorJitter(0.4, 0.4, 0.4, 0.2),
        A.RandomBrightnessContrast(0.2, 0.2)
    ], p=0.8),
    A.CoarseDropout(
        max_holes=8,
        max_height=IMG_SIZE // 10,
        max_width=IMG_SIZE // 10,
        p=0.5
    ),
    A.Normalize((0.485, 0.456, 0.406),
                (0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_tfms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize((0.485, 0.456, 0.406),
                (0.229, 0.224, 0.225)),
    ToTensorV2()
])

# =========================================================
# 6. Dataset
# =========================================================
class SceneDataset(Dataset):
    def __init__(self, df, img_dir, transforms):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        file_id = self.df.loc[idx, "file_id"]
        img_path = os.path.join(self.img_dir, f"{file_id}.jpg")

        image = cv2.imread(img_path)
        if image is None:
            return self.__getitem__((idx + 1) % len(self.df))

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = self.transforms(image=image)["image"]

        if "class" in self.df.columns:
            return image, self.df.loc[idx, "class"]
        return image, file_id

# =========================================================
# 7. Dataloaders
# =========================================================
train_loader = DataLoader(
    SceneDataset(train_df, TRAIN_DIR, train_tfms),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    SceneDataset(val_df, TRAIN_DIR, val_tfms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# =========================================================
# 8. Model
# =========================================================
model = timm.create_model(
    "resnet50",
    pretrained=True,
    num_classes=NUM_CLASSES
).to(DEVICE)

# =========================================================
# 9. Loss, Optimizer, Scheduler
# =========================================================
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(df["class"]),
    y=df["class"]
)
weights = torch.tensor(weights, dtype=torch.float).to(DEVICE)

criterion = nn.CrossEntropyLoss(
    weight=weights,
    label_smoothing=0.1
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

# =========================================================
# 10. Training Loop + Early Stopping
# =========================================================
best_f1 = 0
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()

    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            preds.extend(outputs.argmax(1).cpu().numpy())
            targets.extend(labels.numpy())

    f1 = f1_score(targets, preds, average="macro")
    scheduler.step()

    print(f"Epoch {epoch+1} | Macro F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pth")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping")
            break

print("Best Validation F1:", best_f1)

# =========================================================
# 11. Inference + TTA
# =========================================================
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

def predict_tta(model, image):
    preds = []
    for flip in [False, True]:
        img = image.clone()
        if flip:
            img = torch.flip(img, dims=[3])
        preds.append(torch.softmax(model(img), dim=1))
    return torch.mean(torch.stack(preds), dim=0)

predictions = []

with torch.no_grad():
    for _, row in tqdm(df_test.iterrows(), total=len(df_test)):
        file_id = row["file_id"]
        img_path = os.path.join(TEST_DIR, f"{file_id}.jpg")

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = val_tfms(image=image)["image"].unsqueeze(0).to(DEVICE)

        probs = predict_tta(model, image)
        predictions.append(probs.argmax(1).item())

# =========================================================
# 12. Submission
# =========================================================
submission = pd.DataFrame({
    "file_id": df_test["file_id"],
    "class": predictions
})

submission.to_csv("submission.csv", index=False)
print("submission.csv ready")
